<a href="https://colab.research.google.com/github/fralfaro/ICS40125/blob/main/docs/labs/lab_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# ICS40125 - Laboratorio N°03





**Objetivo**: Aplicar técnicas avanzadas de manipulación y análisis de datos con pandas sobre un conjunto real de datos de contenido de Netflix, reforzando buenas prácticas y métodos eficientes sin recurrir a `groupby`, `merge`, `pivot`, ni `join`.



**Dataset**:

Trabajaremos con el archivo `netflix_titles.csv`, que contiene información sobre los títulos disponibles en la plataforma Netflix hasta el año 2021.

| Variable       | Clase     | Descripción                                                                 |
|----------------|-----------|------------------------------------------------------------------------------|
| show_id        | caracter  | Identificador único del título en el catálogo de Netflix.                   |
| type           | caracter  | Tipo de contenido: 'Movie' o 'TV Show'.                                     |
| title          | caracter  | Título del contenido.                                                       |
| director       | caracter  | Nombre del director (puede ser nulo).                                       |
| cast           | caracter  | Lista de actores principales (puede ser nulo).                              |
| country        | caracter  | País o países donde se produjo el contenido.                                |
| date_added     | fecha     | Fecha en la que el título fue agregado al catálogo de Netflix.              |
| release_year   | entero    | Año de lanzamiento original del título.                                     |
| rating         | caracter  | Clasificación por edad (por ejemplo: 'PG-13', 'TV-MA').                      |
| duration       | caracter  | Duración del contenido (minutos o número de temporadas para series).        |
| listed_in      | caracter  | Categorías o géneros en los que está clasificado el contenido.              |
| description    | caracter  | Breve sinopsis del contenido.                                               |




In [10]:
import pandas as pd

# Cargar datos
df = pd.read_csv('https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/netflix_titles.csv')
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...



### Parte 1: Limpieza y preparación

1. Revisar y describir el dataset:

   * ¿Cuántas filas y columnas tiene?
   * ¿Qué tipos de datos hay?
   * ¿Cuántos valores nulos hay por columna?

2. Transformar la columna `date_added` a tipo fecha.

3. Crear columnas auxiliares con `assign`:

   * Año (`year_added`)
   * Mes (`month_added`)



In [12]:
print("Número de filas y columnas:", df.shape)
print("\nTipos de datos:\n")
df.info()
print("\nValores nulos por columna:\n", df.isnull().sum())

# 2. Transformar la columna date_added a tipo fecha
df['date_added'] = pd.to_datetime(df['date_added'], format='mixed')

# 3. Crear columnas auxiliares con assign:
#    Año (year_added)
#    Mes (month_added)
df = df.assign(
    year_added=df['date_added'].dt.year,
    month_added=df['date_added'].dt.month
)

print("\nDataFrame después de las transformaciones (primeras 5 filas):\n")
print(df.head())

Número de filas y columnas: (8807, 12)

Tipos de datos:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB

Valores nulos por columna:
 show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       

## Parte 2: Técnicas avanzadas de pandas

4. Utilizar `.loc` para seleccionar películas (`type == 'Movie'`) que fueron agregadas después del año 2018.

5. Utilizar `str.contains()` y `str.extract()`:

   * Filtrar títulos que contienen la palabra 'love' (sin distinguir mayúsculas/minúsculas).
   * Extraer la duración en minutos para las películas desde la columna `duration`.

6. Aplicar `explode()` sobre la columna `listed_in` para obtener una fila por cada género.

7. Obtener un top 10 de géneros más frecuentes utilizando `value_counts()`.

8. Aplicar `where()` y `mask()` para marcar las películas de más de 120 minutos como contenido largo en una nueva columna.

9. Utilizar `.loc` para filtrar películas que cumplen con:

   * Más de 100 minutos de duración.
   * Rating igual a `'R'`.
   * País igual a `'United States'`.

10. Utilizar `.style` para formatear visualmente el top 10 de películas más largas.

In [14]:
# 4. Utilizar .loc para seleccionar películas (type == 'Movie') que fueron agregadas después del año 2018.
movies_after_2018 = df.loc[(df['type'] == 'Movie') & (df['year_added'] > 2018)]
print("\n4. Películas (Movies) agregadas después de 2018 (primeras 5 filas):\n")
print(movies_after_2018.head())

# 5. Utilizar str.contains() y str.extract():
#    - Filtrar títulos que contienen la palabra 'love' (sin distinguir mayúsculas/minúsculas).
love_titles = df[df['title'].str.contains('love', case=False, na=False)]
print("\n5a. Títulos que contienen 'love' (primeras 5 filas):\n")
print(love_titles[['title', 'type']].head())

#    - Extraer la duración en minutos para las películas desde la columna duration.
movie_duration = df[df['type'] == 'Movie'].copy()
movie_duration['duration_minutes'] = movie_duration['duration'].str.extract('(\d+)\smin').astype(float)
print("\n5b. Duración en minutos para películas (primeras 5 filas):\n")
print(movie_duration[['title', 'duration', 'duration_minutes']].head())

# 6. Aplicar explode() sobre la columna listed_in para obtener una fila por cada género.
# Primero, limpiar y dividir la columna 'listed_in'
df_exploded = df.assign(listed_in=df['listed_in'].str.split(', ')).explode('listed_in')
print("\n6. DataFrame con 'listed_in' explotado (primeras 5 filas):\n")
print(df_exploded[['title', 'listed_in']].head())

# 7. Obtener un top 10 de géneros más frecuentes utilizando value_counts().
top_10_genres = df_exploded['listed_in'].value_counts().head(10)
print("\n7. Top 10 géneros más frecuentes:\n")
print(top_10_genres)

# 8. Aplicar where() y mask() para marcar las películas de más de 120 minutos como contenido largo en una nueva columna.
# Aseguramos que 'duration_minutes' existe y es numérica para todas las películas (solo tipo 'Movie' tendrá valores, otros NaN)
df['duration_minutes'] = df['duration'].str.extract('(\d+)\smin').astype(float)
df['content_length'] = df['duration_minutes'].apply(lambda x: 'Largo' if x > 120 else 'Corto' if pd.notna(x) else 'N/A')
# Opcional: usando mask y where para esta misma lógica
# df['content_length'] = 'N/A' # Default para no-peliculas o nulos
# df.loc[df['type'] == 'Movie', 'content_length'] = 'Corto'
# df.loc[(df['type'] == 'Movie') & (df['duration_minutes'] > 120), 'content_length'] = 'Largo'
print("\n8. Películas con 'content_length' (primeras 5 filas):\n")
print(df[['title', 'type', 'duration_minutes', 'content_length']].head())

# 9. Utilizar .loc para filtrar películas que cumplen con:
#    - Más de 100 minutos de duración.
#    - Rating igual a 'R'.
#    - País igual a 'United States'.
filtered_movies = df.loc[
    (df['type'] == 'Movie') &
    (df['duration_minutes'] > 100) &
    (df['rating'] == 'R') &
    (df['country'] == 'United States')
]
print("\n9. Películas filtradas por duración > 100 min, Rating 'R', País 'United States' (primeras 5 filas):\n")
print(filtered_movies[['title', 'duration_minutes', 'rating', 'country']].head())

# 10. Utilizar .style para formatear visualmente el top 10 de películas más largas.
top_10_longest_movies = df[df['type'] == 'Movie'].sort_values(by='duration_minutes', ascending=False).head(10)
print("\n10. Top 10 películas más largas (formato .style):\n")
# Para mostrar el estilo, se imprime directamente el objeto styled DataFrame
top_10_longest_movies_styled = top_10_longest_movies[['title', 'duration_minutes', 'release_year', 'rating']].style.background_gradient(cmap='Blues')
# Si esto se ejecuta en un notebook, el estilo se renderizará automáticamente.
# Para una salida de consola, `_repr_html_()` es útil, pero para el agente, simplemente se devuelve el objeto styled.
# print(top_10_longest_movies_styled.to_html()) # Descomentar para ver la salida HTML si es necesario
# En un entorno interactivo de Colab, simplemente mostrar la variable producirá la tabla formateada.
display(top_10_longest_movies_styled)



4. Películas (Movies) agregadas después de 2018 (primeras 5 filas):

   show_id   type                             title  \
0       s1  Movie              Dick Johnson Is Dead   
6       s7  Movie  My Little Pony: A New Generation   
7       s8  Movie                           Sankofa   
9      s10  Movie                      The Starling   
12     s13  Movie                      Je Suis Karl   

                         director  \
0                 Kirsten Johnson   
6   Robert Cullen, José Luis Ucha   
7                    Haile Gerima   
9                  Theodore Melfi   
12            Christian Schwochow   

                                                 cast  \
0                                                 NaN   
6   Vanessa Hudgens, Kimiko Glenn, James Marsden, ...   
7   Kofi Ghanaba, Oyafunmike Ogunlano, Alexandra D...   
9   Melissa McCarthy, Chris O'Dowd, Kevin Kline, T...   
12  Luna Wedler, Jannis Niewöhner, Milan Peschel, ...   

                                 

<>:14: SyntaxWarning: invalid escape sequence '\d'
<>:31: SyntaxWarning: invalid escape sequence '\d'
<>:14: SyntaxWarning: invalid escape sequence '\d'
<>:31: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_482/397070357.py:14: SyntaxWarning: invalid escape sequence '\d'
  movie_duration['duration_minutes'] = movie_duration['duration'].str.extract('(\d+)\smin').astype(float)
/tmp/ipykernel_482/397070357.py:31: SyntaxWarning: invalid escape sequence '\d'
  df['duration_minutes'] = df['duration'].str.extract('(\d+)\smin').astype(float)


,title,duration_minutes,release_year,rating
4253,Black Mirror: Bandersnatch,312.000000,2018,TV-MA
717,Headspace: Unwind Your Mind,273.000000,2021,TV-G
2491,The School of Mischief,253.000000,1973,TV-14
2487,No Longer kids,237.000000,1979,TV-14
2484,Lock Your Girls In,233.000000,1982,TV-PG
2488,Raya and Sakina,230.000000,1984,TV-14
166,Once Upon a Time in America,229.000000,1984,R
7932,Sangam,228.000000,1964,TV-14
1019,Lagaan,224.000000,2001,PG
4573,Jodhaa Akbar,214.000000,2008,TV-14




### Pregunta Desafío

11. ¿Cuáles son las combinaciones más frecuentes de género y rating en el dataset?
    (Sugerencia: utilizar `value_counts` con `subset=["genre", "rating"]` después de aplicar `explode()`).



### Bonus: Análisis de duplicados y limpieza

12. ¿Existen películas con el mismo nombre (`title`) pero con distinto año de lanzamiento (`release_year`)?
13. ¿Cuántos títulos únicos hay en total en la columna `title`?





In [16]:
# 11. ¿Cuáles son las combinaciones más frecuentes de género y rating en el dataset?
# (Sugerencia: utilizar value_counts con subset=["genre", "rating"] después de aplicar explode()).

# df_exploded ya tiene la columna 'listed_in' explotada.
# Necesitamos asegurarnos de que la columna 'rating' también esté presente en df_exploded.
# Al hacer explode, otras columnas se replican para cada nuevo género.
# Así que, podemos usar directamente df_exploded para este análisis.

print("\n11. Combinaciones más frecuentes de género y rating:\n")
genre_rating_combinations = df_exploded[['listed_in', 'rating']].value_counts().head(10)
print(genre_rating_combinations)


### Bonus: Análisis de duplicados y limpieza

# 12. ¿Existen películas con el mismo nombre (title) pero con distinto año de lanzamiento (release_year)?
print("\n12. Películas con el mismo título pero diferente año de lanzamiento:\n")
duplicate_titles = df[df.duplicated(subset=['title'], keep=False)]

# Filtrar solo aquellos que tienen el mismo título pero diferente release_year
different_release_year_same_title = duplicate_titles.groupby('title')['release_year'].nunique()
different_release_year_same_title = different_release_year_same_title[different_release_year_same_title > 1]

if not different_release_year_same_title.empty:
    # Para mostrar los títulos y sus respectivos años de lanzamiento
    titles_with_multiple_years = df[df['title'].isin(different_release_year_same_title.index)].sort_values(by=['title', 'release_year'])
    print(titles_with_multiple_years[['title', 'release_year']].drop_duplicates())
else:
    print("No se encontraron películas con el mismo título y diferente año de lanzamiento.")

# 13. ¿Cuántos títulos únicos hay en total en la columna title?
print("\n13. Cantidad de títulos únicos en el dataset:")
num_unique_titles = df['title'].nunique()
print(num_unique_titles)



11. Combinaciones más frecuentes de género y rating:

listed_in               rating
International Movies    TV-MA     1130
                        TV-14     1065
Dramas                  TV-MA      830
International TV Shows  TV-MA      714
Dramas                  TV-14      693
International TV Shows  TV-14      472
Comedies                TV-14      465
TV Dramas               TV-MA      434
Comedies                TV-MA      431
Dramas                  R          375
Name: count, dtype: int64

12. Películas con el mismo título pero diferente año de lanzamiento:

No se encontraron películas con el mismo título y diferente año de lanzamiento.

13. Cantidad de títulos únicos en el dataset:
8807
